# 04 — Reranking

*Level 2 — Advanced RAG*

## Objective
Retrieve wide with hybrid search, rerank the candidates with a cross-encoder, and measure — again, honestly — whether it improves ranking quality on this corpus.

**Requirement:** `uv sync --extra sentence-transformers` (adds torch).


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))
sys.path.insert(0, str(LEVEL_DIR / "query-transformations"))
sys.path.insert(0, str(LEVEL_DIR / "metadata-filtering"))
sys.path.insert(0, str(LEVEL_DIR / "context-compression"))


In [2]:
import random
from common.dataset import prepare
from retrieval.dense import DenseRetriever
from retrieval.sparse import BM25Retriever
from bm25_vector import HybridRetriever
from reranking.cross_encoder import CrossEncoderReranker
from evaluation.recall_at_k import recall_at_k
from evaluation.ndcg import ndcg_at_k

data = prepare()
corpus_texts = {d: data.corpus_text(d) for d in data.doc_ids()}
dense = DenseRetriever.from_corpus(corpus_texts)
sparse = BM25Retriever.from_corpus(corpus_texts)
hybrid = HybridRetriever(dense, sparse)

print("Loading cross-encoder (downloads ~80MB the first time)...")
reranker = CrossEncoderReranker()
print(f"model: {reranker.model_name}")


Loading cross-encoder (downloads ~80MB the first time)...


/Users/yessinezghal/Desktop/learn/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9669.39it/s]

model: cross-encoder/ms-marco-MiniLM-L-6-v2


## Before/after on a sample of queries

Reranking every candidate for all 300 queries works the same way — this notebook uses a 60-query sample so it runs in well under a minute; see `evaluation/` + `examples/advanced_pipeline.py` to scale this up.


In [3]:
rng = random.Random(0)
sample_qids = rng.sample(list(data.queries.keys()), 60)

hybrid_results, reranked_results = {}, {}
for qid in sample_qids:
    q = data.queries[qid]
    candidates = hybrid.search(q, top_k=20)
    hybrid_results[qid] = [d for d, _ in candidates]
    cand_texts = [(d, corpus_texts[d]) for d, _ in candidates]
    reranked_results[qid] = [d for d, _ in reranker.rerank(q, cand_texts, top_k=10)]

sample_qrels = {qid: data.qrels[qid] for qid in sample_qids}
print(f"{'k':>4} {'recall before':>14} {'recall after':>14} {'ndcg before':>13} {'ndcg after':>12}")
for k in (1, 3, 5, 10):
    rb = recall_at_k(hybrid_results, sample_qrels, k)
    ra = recall_at_k(reranked_results, sample_qrels, k)
    nb_ = ndcg_at_k(hybrid_results, sample_qrels, k)
    na = ndcg_at_k(reranked_results, sample_qrels, k)
    print(f"{k:>4} {rb:>14.3f} {ra:>14.3f} {nb_:>13.3f} {na:>12.3f}")


   k  recall before   recall after   ndcg before   ndcg after
   1          0.717          0.700         0.717        0.700
   3          0.817          0.817         0.771        0.764
   5          0.867          0.867         0.792        0.784
  10          0.900          0.917         0.803        0.801


## One query, before vs. after reranking


In [4]:
sample_qid = sample_qids[0]
q = data.queries[sample_qid]
relevant = set(data.qrels[sample_qid])
print(f"Query: {q!r}  (relevant: {relevant})\n")

print("Hybrid order (before rerank):")
for d in hybrid_results[sample_qid][:5]:
    print(f"  {'<-- relevant' if d in relevant else '':<13}{d}")
print("\nReranked order (after cross-encoder):")
for d in reranked_results[sample_qid][:5]:
    print(f"  {'<-- relevant' if d in relevant else '':<13}{d}")


Query: 'PDPN promotes efficient motility along stromal surfaces by activating the C-type lectin receptor to rearrange the actin cytoskeleton in dendritic cells.'  (relevant: {'7370282'})

Hybrid order (before rerank):
  <-- relevant 7370282
               8997410
               2177022
               20313748
               17194716

Reranked order (after cross-encoder):
  <-- relevant 7370282
               4447055
               36345185
               12885341
               2177022


## What I observed

Reranking gave a **small, mixed** result here: essentially flat Recall@3/@5, a slight dip at @1, a slight gain at @10. That's a real, measured outcome — not the universal win reranking gets marketed as.

The likely cause: `cross-encoder/ms-marco-MiniLM-L-6-v2` is trained on **general web search** relevance (MS MARCO), not biomedical claim verification (scifact's actual domain). A cross-encoder's accuracy comes from deep query-document interaction, which is only as good as how well its training distribution matches yours — an off-the-shelf reranker fine-tuned on the wrong domain can easily fail to beat a domain-appropriate retriever it's supposed to improve on.

`reranking/bge_reranker.py` provides `BGEReranker` as a drop-in alternative (`BAAI/bge-reranker-base`) with the identical interface — worth comparing, at the cost of a larger (~1.1GB) model download.

**Lesson:** always measure a reranker on your own eval set before trusting it in production — Level 2's whole premise is that these techniques are *tunable*, not automatically beneficial.

## Next

[05 — Query Transformations](./05_query_transformations.ipynb)
